# Steamroller — a quantitative teardown 🔬
### The carry premium by rate bucket · Newey-West t · the negative-skew crash · vol-managed carry · the UIRP null

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Crash risk?: Severe](https://img.shields.io/badge/Crash_risk%3F-Severe-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim with its standard error.* The steelman is §8.2, the FX carry trade: UIRP fails, so high-rate currencies earn a premium. We prove the engine on a synthetic G10 with a baked premium and risk-off crashes, then read the real verdict off the shared G10 tape (OECD MEI short rates + FX, 270 months 2001-08 → 2024-01, as-of 2024-01-31, fingerprint `ef7450ae792e` — the same cache Study 36 runs on).

> ⚠️ **Not investment advice.** The executed cells run on the synthetic control; every real-tape number is quoted from [`../docs/results.md`](../docs/results.md), sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from steamroller import data, carry, strategy, decompose, extension

# Offline synthetic G10: a CARRY PREMIUM tape (high-rate currencies out-earn, punctuated by sticky
# risk-off crashes) and a full-UIRP NULL. The real G10 verdict (OECD rates + FX, 2001-2024, shared
# with Study 36) is quoted from ../docs/results.md.
xr,  rates,  truth = data.synthetic_carry(carry_strength=0.9, seed=27)   # the carry-premium tape
xr0, rates0, _     = data.synthetic_carry(carry_strength=0.0, seed=27)   # the full-UIRP null
print(f"{truth.n_ccy} currencies x {truth.n_months} months | baked carry_strength={truth.carry_strength} | null=0")


9 currencies x 600 months | baked carry_strength=0.9 | null=0


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — is the premium real? | 🟡 `WEAK` | Real 2001–2024 G10: bucket spread **+2.8%/yr**, book **+0.8%/yr** net at HAC *t* **+0.9** (gross **+1.0**), Sharpe **+0.22** with bootstrap 95% CI **[−0.25, +0.72]** — positive, the right slope, but this sample can't reject zero. The control proves the machine (premium **+2.1%/yr**, *t* **+2.8**; null flat, *t* **-0.2**). |
| **Tradability** | 🟡 `FRAGILE` | Genuinely low turnover (**0.55×/yr** real — survives even 100 bp), but the edge is thin and the tail fat: real skew **−0.90**, worst month **−5.8%**, drawdown **−15%**. |
| **Crash risk?** | ⚪ `Severe` | The crash is a sudden risk-off jump that vol-targeting can't forecast — on the real tape the overlay *cuts* the Sharpe (**+0.20 → +0.11**) and *deepens* the drawdown (**−15% → −26%**). |

> **In one sentence:** the carry premium is durable in the literature but thin on the post-2000 tape, cheap to run — and it is compensation for a sharply negative-skewed crash that the desk's usual vol overlay cannot dodge.

*(This notebook executes on the synthetic control; the real G10 numbers are quoted from [`../docs/results.md`](../docs/results.md), reproducible offline via `examples/verify.py`.)*

## Beat 1 · The claim, precisely

UIRP: $(1+r_d) = \mathbb{E}_t[S_{t+T}]/S_t \cdot (1+r_f)$ — the high-rate currency should depreciate to offset its yield. Empirically it doesn't, so the excess return of holding currency $i$ funded in the base is $x_i \approx (r_i - r_{\text{base}}) + \Delta\log S_i > 0$ for high $r_i$. The carry portfolio is long the top-rate, short the bottom-rate tercile, **weights set on the prior month-end's rates** — and the bucket diagnostic sorts on the same lagged rate, so no diagnostic ever uses month-$t$ information to explain month-$t$'s return. The synthetic bakes partial UIRP (the premium) plus sticky risk-off crashes; `carry_strength = 0` is full UIRP (the null).

In [2]:
pb = carry.carry_premium_by_bucket(xr, rates)   # sorted on the lagged rate
print(f"high-minus-low rate bucket spread {pb['hml_ann_pct']:+.1f}%/yr (synthetic); "
      f"null {carry.carry_premium_by_bucket(xr0, rates0)['hml_ann_pct']:+.1f}%/yr")
print('real G10 (../docs/results.md): +2.8%/yr')

high-minus-low rate bucket spread +4.2%/yr (synthetic); null -0.1%/yr
real G10 (../docs/results.md): +2.8%/yr


## Beat 2 · So what?

Carry's defining feature is its **skewness**, not its mean. Brunnermeier, Nagel & Pedersen (2008) tie carry crashes to the sudden unwinding of crowded, leveraged positions when funding liquidity dries up — a jump, correlated across all carry pairs at once. So the premium is a risk premium for a crash, and the open questions are: is it significant, how fat is the tail, and does any cheap overlay shrink it. Beats 4–6 answer all three.

## Beat 3 · Pre-registered protocol

1. **Premium** (`carry.carry_premium_by_bucket` on the lagged rate, `decompose.premium_tstat`): bucket spread + HAC *t* on the real tape. `REAL` ⇔ positive *and* *t* > 2; positive but *t* ≤ 2 ⇒ `WEAK`.
2. **Crash** (`decompose.crash_profile`, `downside_concentration`): skew, worst months, drawdown.
3. **Risk management** (`extension.crash_comparison`, both books on the shared post-burn-in window): vol-managed vs plain. `Severe` ⇔ the overlay fails to shrink the tail.
4. **Null:** full UIRP collapses the premium (machinery check).

**Verdict logic:** the stamps follow the real-tape numbers; the control only certifies the tools.

## Beat 4 · The teardown

### 4a · Premium and t — control, null, and the real tape

In [3]:
for label, (x, rt) in [('carry', (xr, rates)), ('null', (xr0, rates0))]:
    pt = decompose.premium_tstat(x, rt, cost_bps=10.0); cmp = strategy.compare(x, rt, cost_bps=10.0)
    print(f"{label:6s}: premium {pt['mean_ann_pct']:+.1f}%/yr (HAC t {pt['t_stat']:+.1f}), "
          f"Sharpe {cmp['sharpe']:+.2f}, turnover {cmp['turnover_ann']:.1f}x")
print()
print('real G10: premium +0.8%/yr net (HAC t +0.9; gross t +1.0), '
      'Sharpe +0.22 (bootstrap 95% CI [−0.25, +0.72]), turnover 0.55x/yr')
# the control's constant per-currency rates mean its book never rebalances (turnover 0.0x):
# the cost claim is only exercised on the real tape, where the Sharpe still holds at 100 bp.

carry : premium +2.1%/yr (HAC t +2.8), Sharpe +0.60, turnover 0.0x
null  : premium -0.1%/yr (HAC t -0.2), Sharpe -0.03, turnover 0.0x

real G10: premium +0.8%/yr net (HAC t +0.9; gross t +1.0), Sharpe +0.22 (bootstrap 95% CI [−0.25, +0.72]), turnover 0.55x/yr


> 💡 **In plain words.** The tools find the premium when it's there (control) and nothing when it isn't (null). Pointed at the real market, they find *something* — the right slope, a positive book — but at a *t* of 0.9 you couldn't tell it from luck on this sample alone. That's a `WEAK`, said plainly.

### 4b · The crash profile and downside concentration

In [4]:
cr = decompose.crash_profile(xr, rates, cost_bps=10.0)
dc = decompose.downside_concentration(xr, rates, cost_bps=10.0, k=5)
print(f"synthetic: skew {cr['skew']:+.2f}, worst month {cr['worst_month_pct']:+.1f}%, worst-5 {cr['worst5_months_mean_pct']:+.1f}%, max drawdown {cr['max_drawdown_pct']:.0f}%")
print(f"  the worst 5 months carry {dc['worst_k_share_of_losses']:.0%} of all losing-month losses -- crash-concentrated")
print('real G10 : skew −0.90, worst month −5.8% (Oct-2008), worst-5 −3.3%, '
      'max drawdown −15%; worst-5 share 17% of losses (../docs/results.md)')

synthetic: skew -1.54, worst month -4.3%, worst-5 -3.8%, max drawdown -28%
  the worst 5 months carry 11% of all losing-month losses -- crash-concentrated
real G10 : skew −0.90, worst month −5.8% (Oct-2008), worst-5 −3.3%, max drawdown −15%; worst-5 share 17% of losses (../docs/results.md)


> 💡 **In plain words.** Carry doesn't lose a little often; it loses a lot, rarely, all at once. The mean looks like a smooth yield, but a handful of months hold most of the pain — the signature of a risk premium for a tail event. On the real tape those months have names: September–October 2008, March 2020.

### 4c · Vol-management does not tame the tail — and on the real tape it backfires

In [5]:
cc = extension.crash_comparison(xr, rates, cost_bps=10.0)   # both books on the shared post-burn-in window
import pandas as pd; display(pd.DataFrame({k: cc[k] for k in ('plain', 'managed')}).T.round(2))
print(f"(shared window: {cc['n_months']} months -- the plain Sharpe here differs from the "
      'full-sample headline because the managed book needs a 12-month vol burn-in)')
print('synthetic: Sharpe up, drawdown not down.')
print('real G10 : Sharpe +0.20 -> +0.11 (DOWN), drawdown −15% -> −26% '
      '(DEEPER) -- trailing vol reads the calm 2003-07 build-up as safety and levers into the 2008 jump.')

,sharpe,skew,worst_month_pct,max_drawdown_pct
plain,0.6800,-1.5700,-4.3200,-28.2700
managed,0.9500,-1.0100,-10.3600,-43.5300


(shared window: 588 months -- the plain Sharpe here differs from the full-sample headline because the managed book needs a 12-month vol burn-in)
synthetic: Sharpe up, drawdown not down.
real G10 : Sharpe +0.20 -> +0.11 (DOWN), drawdown −15% -> −26% (DEEPER) -- trailing vol reads the calm 2003-07 build-up as safety and levers into the 2008 jump.


## Beat 5 · The verdict

- **A thin premium** (4a): real tape +0.8%/yr at *t* +0.9, CI [−0.25, +0.72] — positive but not significant; control +2.1%/yr (*t* +2.8), null flat → `WEAK`.
- **Severe, concentrated tail** (4b): real skew −0.90, worst month −5.8%, drawdown −15%.
- **Unhedgeable by vol-targeting** (4c): on the real tape the overlay cuts the Sharpe and deepens the drawdown.

> **Signal `WEAK` · Tradability `FRAGILE` · Crash risk? `Severe`.**

## Beat 6 · Could you trade it?

- **Cheap to run** — real turnover 0.55×/yr; the cost sweep barely moves the Sharpe (+0.24 at 0 bp → +0.08 at 100 bp). Cost is not the binding constraint.
- **Thin and crash-prone is** — +0.8%/yr of edge against a −5.8% month and a drawdown correlated across every carry pair you'd hold.
- **Vol-targeting fails** — it levers you *into* the jump (4c).

Tradability **`FRAGILE`**; crash **`Severe`**.

## Beat 7 · Going further

### 7a · Worked complement — why risk management doesn't dodge the steamroller
Plain vs vol-managed carry, side by side; the Sharpe rises but the drawdown doesn't fall.

In [6]:
cc = extension.crash_comparison(xr, rates, cost_bps=10.0)
for k in ['plain', 'managed']:
    p = cc[k]; print(f"{k:8s}: Sharpe {p['sharpe']:+.2f}, skew {p['skew']:+.2f}, worst month {p['worst_month_pct']:+.1f}%, max drawdown {p['max_drawdown_pct']:.0f}%")
print('Real G10 (../docs/extension.md): harsher still -- Sharpe +0.20 -> +0.11, '
      'worst month −5.8% -> −9.6%, drawdown −15% -> −26%.')

plain   : Sharpe +0.68, skew -1.57, worst month -4.3%, max drawdown -28%
managed : Sharpe +0.95, skew -1.01, worst month -10.4%, max drawdown -44%
Real G10 (../docs/extension.md): harsher still -- Sharpe +0.20 -> +0.11, worst month −5.8% -> −9.6%, drawdown −15% -> −26%.


**The result.** Vol-targeting — the overlay that earned [Study 16](../../16-storm-shy/) the desk's only green and tamed momentum's crash in [Study 24](../../24-stampede/) — *fails* on carry. On the control it lifts the standalone Sharpe while deepening the tail; on the real tape it is a strict loss on every axis, because trailing vol reads the calm 2003–07 carry build-up as safety and levers into the 2008 jump. That is the cleanest statement of *why* carry's crash is `Severe`: it is the one tail on this desk that a trailing-volatility forecast cannot see. The honest defences — options, a risk-off switch, cross-asset diversification — are forks, not a vol target. Full run in [`../docs/extension.md`](../docs/extension.md).

### 7b · Other forks
- **Diversification, measured** — [Study 36 (Greenback)](../../36-greenback/) runs the carry⊕momentum combo on this exact tape (same fingerprint): the crash is cushioned, the Sharpe is not lifted.
- **Risk-off conditioning** — gate the book on a VIX / drawdown / funding-stress signal (Brunnermeier-Nagel-Pedersen) and see if it dodges the jump without killing the premium.
- **Options tail hedge** — price the put protection against the carry premium; is the net still positive?
- **A longer tape** — our rates source starts in 2001; forward points would reach the fat carry decades and test whether `WEAK` is decay or sample.

PRs welcome — extend the tape, or build a tail hedge that earns its keep.